# 01. Medication Event Dataset Schema

This notebook documents the master dataset for INTELLIMED medication events. The schema supports device logs, backend events, adherence tracking, and future ML feature engineering.

The dataset is intentionally designed for project analytics and not for medical decision-making.

In [ ]:
from pathlib import Path
import pandas as pd

schema = [
    {
        "column_name": "patient_id",
        "data_type": "string",
        "meaning": "Unique identifier for the patient",
        "allowed_values": "Any non-empty patient code",
        "required": True,
        "source": "device / backend registration",
        "future_ml": True,
    },
    {
        "column_name": "medicine_id",
        "data_type": "string",
        "meaning": "Unique identifier for the medicine",
        "allowed_values": "Any non-empty medicine code",
        "required": True,
        "source": "backend medication registry",
        "future_ml": True,
    },
    {
        "column_name": "medicine_name",
        "data_type": "string",
        "meaning": "Human readable medicine name",
        "allowed_values": "Free text, normalized when possible",
        "required": True,
        "source": "backend prescription data",
        "future_ml": True,
    },
    {
        "column_name": "compartment_id",
        "data_type": "string",
        "meaning": "Hardware compartment slot in the box",
        "allowed_values": "Device-specific labels such as C1, C2",
        "required": True,
        "source": "ESP32 event logs",
        "future_ml": True,
    },
    {
        "column_name": "scheduled_date",
        "data_type": "date",
        "meaning": "Date each medication slot was scheduled",
        "allowed_values": "ISO date YYYY-MM-DD",
        "required": True,
        "source": "schedule generation logic",
        "future_ml": True,
    },
    {
        "column_name": "scheduled_time",
        "data_type": "time",
        "meaning": "Planned time for the dose",
        "allowed_values": "HH:MM:SS or HH:MM",
        "required": True,
        "source": "medication schedule configuration",
        "future_ml": True,
    },
    {
        "column_name": "reminder_time",
        "data_type": "time",
        "meaning": "Time when reminder or alert was sent",
        "allowed_values": "HH:MM:SS or HH:MM",
        "required": False,
        "source": "backend reminder service",
        "future_ml": True,
    },
    {
        "column_name": "acknowledged_time",
        "data_type": "time/null",
        "meaning": "Actual time when the patient acknowledged or completed the dose",
        "allowed_values": "HH:MM:SS or null",
        "required": False,
        "source": "device or app event",
        "future_ml": True,
    },
    {
        "column_name": "delay_minutes",
        "data_type": "integer",
        "meaning": "Difference between actual completion and scheduled time in minutes",
        "allowed_values": "Non-negative integer",
        "required": False,
        "source": "event timestamps",
        "future_ml": True,
    },
    {
        "column_name": "dose_status",
        "data_type": "string",
        "meaning": "Outcome for the scheduled dose",
        "allowed_values": "TAKEN, PENDING, MISSED",
        "required": True,
        "source": "project rule engine",
        "future_ml": True,
    },
    {
        "column_name": "day_of_week",
        "data_type": "string",
        "meaning": "Day of week derived from scheduled_date",
        "allowed_values": "MONDAY to SUNDAY",
        "required": True,
        "source": "date derivation",
        "future_ml": True,
    },
    {
        "column_name": "time_period",
        "data_type": "string",
        "meaning": "Time bucket for the dose",
        "allowed_values": "MORNING, AFTERNOON, EVENING, NIGHT",
        "required": True,
        "source": "derived from scheduled_time",
        "future_ml": True,
    },
    {
        "column_name": "frequency",
        "data_type": "string",
        "meaning": "How often the medicine is scheduled",
        "allowed_values": "ONCE_DAILY, TWICE_DAILY, THRICE_DAILY, AS_NEEDED, custom schedule label",
        "required": True,
        "source": "medication configuration",
        "future_ml": True,
    },
    {
        "column_name": "dose_quantity",
        "data_type": "float",
        "meaning": "Quantity associated with the dose",
        "allowed_values": "Non-negative number",
        "required": False,
        "source": "prescription or device config",
        "future_ml": True,
    },
    {
        "column_name": "previous_missed_doses",
        "data_type": "integer",
        "meaning": "Missed dose count before the current event",
        "allowed_values": "Non-negative integer",
        "required": False,
        "source": "rolling adherence history",
        "future_ml": True,
    },
    {
        "column_name": "previous_taken_doses",
        "data_type": "integer",
        "meaning": "Taken dose count before the current event",
        "allowed_values": "Non-negative integer",
        "required": False,
        "source": "rolling adherence history",
        "future_ml": True,
    },
    {
        "column_name": "adherence_percentage",
        "data_type": "float",
        "meaning": "Rolling adherence metric value for patient or medicine",
        "allowed_values": "0.0 to 100.0",
        "required": False,
        "source": "derived analytics",
        "future_ml": True,
    },
    {
        "column_name": "sensor_event",
        "data_type": "string",
        "meaning": "Sensor-level event associated with the dose",
        "allowed_values": "DOOR_OPEN, DOOR_CLOSE, BUTTON_PRESS, SENSOR_TIMEOUT, UNKNOWN",
        "required": False,
        "source": "ESP32 telemetry",
        "future_ml": True,
    },
]

schema_df = pd.DataFrame(schema)
print(schema_df.to_string(index=False))

## Dose status definition

The project standardizes dose states as:

- `TAKEN`: the dose was completed or acknowledged within the configured reminder and grace-period rules.
- `PENDING`: the dose is scheduled but has not yet reached its decision point or was not yet marked overdue.
- `MISSED`: the configured reminder and grace period elapsed without the dose being taken.

This state is operational and project-rule based, not a medical diagnosis.

In [ ]:
status_values = {
    "TAKEN": "Dose acknowledged or completed within the configured window.",
    "PENDING": "Dose is still active and has not reached the overdue threshold.",
    "MISSED": "Configured reminder/grace period expired before the dose was taken.",
}

for status, meaning in status_values.items():
    print(f"{status}: {meaning}")